# Undulator Archiver Range Test

This notebook probes the undulator redline PVs with raw archiver requests.
It starts with a known-bad time range, then keeps bisecting only the failed
windows until a request succeeds or the minimum window size is reached.

The fetch helper here does not use the retry wrapper from
`sparklines_hierarchy.py`; that keeps the timeout behavior visible instead of
masking it behind retries.

In [ ]:
import datetime as dt
import time
from collections import deque
from pathlib import Path
from pprint import pprint

import requests
import yaml

try:
    from app.sparklines_hierarchy import ARCHIVER_URL, LOCAL_TIMEZONE, _format_archive_time
except ImportError:
    from sparklines_hierarchy import ARCHIVER_URL, LOCAL_TIMEZONE, _format_archive_time


def resolve_app_dir() -> Path:
    cwd = Path.cwd()
    if (cwd / "pv_groups.yaml").exists():
        return cwd
    if (cwd / "app" / "pv_groups.yaml").exists():
        return cwd / "app"
    raise FileNotFoundError("Could not find pv_groups.yaml from the current working directory.")


APP_DIR = resolve_app_dir()
PV_GROUPS_PATH = APP_DIR / "pv_groups.yaml"

DEFAULT_START = dt.datetime(2025, 6, 18, 6, 0, 0)
DEFAULT_END = dt.datetime(2025, 6, 18, 12, 0, 0)
DEFAULT_TIMEOUT_SECONDS = 8.0
DEFAULT_MIN_WINDOW = dt.timedelta(minutes=5)
DEFAULT_MAX_DEPTH = 12

print("APP_DIR:", APP_DIR)
print("ARCHIVER_URL:", ARCHIVER_URL)
print("DEFAULT_WINDOW:", DEFAULT_START, "->", DEFAULT_END)


In [ ]:
def load_group_specs(group_name: str = "undulator") -> dict[str, list[str]]:
    with PV_GROUPS_PATH.open("r", encoding="utf-8") as handle:
        payload = yaml.safe_load(handle) or {}

    groups = payload.get("groups", []) or []
    group = next(group for group in groups if group.get("group_name") == group_name)

    specs = {}
    for subgroup_name, subgroup in (group.get("subgroups") or {}).items():
        pv_names = []
        for entry in subgroup.get("pv", []) or []:
            if isinstance(entry, dict) and entry.get("pv_name"):
                pv_names.append(str(entry["pv_name"]).strip())
        if pv_names:
            specs[str(subgroup_name)] = pv_names

    return specs


UNDULATOR_SPECS = load_group_specs()
UNDULATOR_SPECS


In [ ]:
def fetch_archive_window(
    pv_name: str,
    start: dt.datetime,
    end: dt.datetime,
    *,
    timeout: float = DEFAULT_TIMEOUT_SECONDS,
    archiver_url: str = ARCHIVER_URL,
):
    params = {
        "pv": pv_name,
        "from": _format_archive_time(start, local_timezone=LOCAL_TIMEZONE),
        "to": _format_archive_time(end, local_timezone=LOCAL_TIMEZONE),
    }
    started = time.perf_counter()
    response = requests.get(archiver_url, params=params, timeout=timeout)
    elapsed = time.perf_counter() - started
    response.raise_for_status()

    payload = response.json()
    if not isinstance(payload, list) or not payload:
        raise RuntimeError(f"archive returned empty payload for PV {pv_name}")

    data = payload[0].get("data")
    if not isinstance(data, list):
        raise RuntimeError(f"archive payload missing data array for PV {pv_name}")

    return {
        "pv_name": pv_name,
        "start": start,
        "end": end,
        "elapsed_seconds": elapsed,
        "sample_count": len(data),
        "response_bytes": len(response.content),
    }


def split_window(start: dt.datetime, end: dt.datetime) -> tuple[dt.datetime, dt.datetime]:
    return start, start + (end - start) / 2


def bisect_until_success(
    pv_name: str,
    start: dt.datetime,
    end: dt.datetime,
    *,
    timeout: float = DEFAULT_TIMEOUT_SECONDS,
    min_window: dt.timedelta = DEFAULT_MIN_WINDOW,
    max_depth: int = DEFAULT_MAX_DEPTH,
):
    attempts = []
    queue = deque([(start, end, 0)])

    while queue:
        win_start, win_end, depth = queue.popleft()
        duration = win_end - win_start
        attempt = {
            "pv_name": pv_name,
            "depth": depth,
            "start": win_start,
            "end": win_end,
            "duration": duration,
        }

        try:
            payload = fetch_archive_window(pv_name, win_start, win_end, timeout=timeout)
        except Exception as exc:
            attempt["success"] = False
            attempt["error_type"] = type(exc).__name__
            attempt["error"] = str(exc)
            attempts.append(attempt)

            if depth < max_depth and duration > min_window:
                left_start, mid = split_window(win_start, win_end)
                queue.append((left_start, mid, depth + 1))
                queue.append((mid, win_end, depth + 1))
            continue

        attempt.update(payload)
        attempt["success"] = True
        attempts.append(attempt)

    successes = [attempt for attempt in attempts if attempt["success"]]
    failures = [attempt for attempt in attempts if not attempt["success"]]

    return {
        "pv_name": pv_name,
        "attempts": attempts,
        "successes": successes,
        "failures": failures,
        "first_success": successes[0] if successes else None,
        "smallest_success": min(successes, key=lambda item: item["duration"]) if successes else None,
        "smallest_failure": min(failures, key=lambda item: item["duration"]) if failures else None,
    }


def run_undulator_probe(
    start: dt.datetime,
    end: dt.datetime,
    *,
    target_subgroups: list[str] | None = None,
    timeout: float = DEFAULT_TIMEOUT_SECONDS,
    min_window: dt.timedelta = DEFAULT_MIN_WINDOW,
    max_depth: int = DEFAULT_MAX_DEPTH,
):
    results = {}
    for subgroup_name, pv_names in UNDULATOR_SPECS.items():
        if target_subgroups is not None and subgroup_name not in target_subgroups:
            continue
        results[subgroup_name] = [
            bisect_until_success(
                pv_name,
                start,
                end,
                timeout=timeout,
                min_window=min_window,
                max_depth=max_depth,
            )
            for pv_name in pv_names
        ]
    return results


def summarize_probe(results: dict[str, list[dict]]) -> list[dict]:
    summary = []
    for subgroup_name, pv_results in results.items():
        for pv_result in pv_results:
            first_success = pv_result["first_success"]
            smallest_success = pv_result["smallest_success"]
            smallest_failure = pv_result["smallest_failure"]
            summary.append(
                {
                    "subgroup": subgroup_name,
                    "pv_name": pv_result["pv_name"],
                    "attempt_count": len(pv_result["attempts"]),
                    "success_count": len(pv_result["successes"]),
                    "failure_count": len(pv_result["failures"]),
                    "first_success_window": str(first_success["duration"]) if first_success else None,
                    "first_success_samples": first_success["sample_count"] if first_success else None,
                    "smallest_success_window": str(smallest_success["duration"]) if smallest_success else None,
                    "smallest_failure_window": str(smallest_failure["duration"]) if smallest_failure else None,
                    "last_error": pv_result["failures"][-1]["error"] if pv_result["failures"] else None,
                }
            )
    return summary


def find_pv_result(results: dict[str, list[dict]], pv_name: str) -> dict:
    for pv_results in results.values():
        for pv_result in pv_results:
            if pv_result["pv_name"] == pv_name:
                return pv_result
    raise KeyError(pv_name)


In [ ]:
START = DEFAULT_START
END = DEFAULT_END
TARGET_SUBGROUPS = None
TIMEOUT_SECONDS = DEFAULT_TIMEOUT_SECONDS
MIN_WINDOW = DEFAULT_MIN_WINDOW
MAX_DEPTH = DEFAULT_MAX_DEPTH

probe_results = run_undulator_probe(
    START,
    END,
    target_subgroups=TARGET_SUBGROUPS,
    timeout=TIMEOUT_SECONDS,
    min_window=MIN_WINDOW,
    max_depth=MAX_DEPTH,
)

summary_rows = summarize_probe(probe_results)
pprint(summary_rows, sort_dicts=False)


In [ ]:
FOCUS_PV = "SIOC:SYS0:ML04:AO310"

focus_result = find_pv_result(probe_results, FOCUS_PV)
focus_result["attempts"]
